# How to Build a RAG System with Claude + Oracle AI Database Vector Search

This notebook demonstrates how to build a simple Retrieval-Augmented Generation (RAG) pipeline using:
- Claude (Anthropic) for generation
- Oracle AI Database Vector Search for retrieval


You will learn how to:
- Store embeddings in Oracle AI Database
- Run vector similarity search
- Ground Claude’s answers using retrieved context

We will:
1. Set up the environment and install dependencies  
2. Connect to Oracle AI Database  
3. Create a vector table  
4. Generate embeddings  
5. Ingest documents  
6. Perform similarity search  
7. Use retrieved context to answer questions with Claude  


### Step 1: Install dependencies

We install the required Python libraries:
- `oracledb` – Oracle Database driver
- `anthropic` – Claude API client
- `python-dotenv` – load secrets from `.env`
- `pandas`, `numpy`, `matplotlib` – data inspection and visualization


In [1]:
!pip install -U oracledb anthropic python-dotenv pandas numpy sentence-transformers -q

### Prerequisites

This notebook assumes you already have access to an Oracle AI Database instance.

You can use either:

- **Oracle Autonomous Database (recommended for cloud demos / production)**
  - Provision an Autonomous Database on OCI
  - Configure wallet + secure connection

- **Oracle AI Database Free (local)**
  - Run locally via Docker / Podman  
  - Official image: `container-registry.oracle.com/database/free:latest`

Before continuing, make sure:
- Your Oracle AI DB is running
- You can connect using `oracledb`
- Your connection details are set in `.env`

Docs:
- Oracle AI Database Vector Search: https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/

### Step 2: Load environment variables

Create a `.env` file with the following values:

- ANTHROPIC_API_KEY=...
- ORACLE_USER=...
- ORACLE_PASSWORD=...
- ORACLE_DSN=localhost:1521/FREEPDB1


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

print("ORACLE_USER:", os.getenv("ORACLE_USER"))
print("ORACLE_DSN:", os.getenv("ORACLE_DSN"))
print("ANTHROPIC_API_KEY loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


ORACLE_USER: RAG_USER
ORACLE_DSN: localhost:1521/FREEPDB1
ANTHROPIC_API_KEY loaded: True


### Step 3: Connect to Oracle AI Database

You can connect either to:

- **Oracle Autonomous Database (recommended for production & cloud demos)**
- **Oracle AI Database Free running locally (Docker/Podman) for local testing**

Both options use the same Python API (`oracledb`).  
You only need to change the connection configuration.


> For Autonomous DB, make sure your `ORACLE_DSN` points to the Autonomous service
> and that you have configured the wallet or secure connection parameters.

> For example:
> ORACLE_DSN=adb_high


In [3]:
import oracledb
import os

# Works for both:
# - Autonomous DB (with wallet + DSN)
# - Local Oracle AI DB Free (Docker/Podman)
conn = oracledb.connect(
    user=os.getenv("ORACLE_USER"),
    password=os.getenv("ORACLE_PASSWORD"),
    dsn=os.getenv("ORACLE_DSN"),
)

cursor = conn.cursor()
print("Connected to Oracle AI Database")


Connected to Oracle AI Database


### Step 4: Create vector table

We create a table with:
- `content` – the document text
- `embedding VECTOR(1024)` – Oracle native vector type

The vector dimension must match the embedding model output dimension.
Make sure to adjust VECTOR(<DIM>) if you change the embedding model.


In [4]:
cursor.execute("""
BEGIN
  EXECUTE IMMEDIATE 'DROP TABLE documents PURGE';
EXCEPTION
  WHEN OTHERS THEN NULL;
END;
""")

cursor.execute("""
CREATE TABLE documents (
    id NUMBER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    content CLOB,
    embedding VECTOR(1024)
)
""")

conn.commit()
print("Vector table created")


Vector table created


### Step 5: Prepare documents

For demo purposes, we use a small in-memory dataset.
In real applications, this could be:
- PDFs
- Docs
- Web pages
- Product descriptions

In [5]:
docs = [
    "Oracle AI Database provides native vector search for semantic retrieval across enterprise data.",
    "Vector Search enables similarity queries over embeddings stored directly in Oracle tables.",
    "RAG systems combine retrieval with LLMs to reduce hallucinations and improve factual grounding.",
    "Claude is used for generation and reasoning, while embeddings can be produced by a separate model.",
    "Ingesting documents into a vector store enables semantic search over unstructured content.",
    "Top-k retrieval returns the most relevant chunks based on vector distance metrics.",
    "Chunking strategies (size, overlap) significantly impact retrieval quality in RAG systems.",
    "Oracle AI Database integrates vector search with transactional and analytical workloads.",
    "Python applications can connect to Oracle using the oracledb driver for vector operations.",
    "Grounding LLM answers in retrieved context improves reliability and auditability.",
    "RAG pipelines typically include ingestion, indexing, retrieval, and generation steps.",
    "Vector indexes can accelerate similarity search for large corpora.",
    "Developers should monitor similarity score distributions to tune retrieval thresholds.",
    "Evaluating retrieval quality is crucial before optimizing LLM prompts.",
    "Domain-specific corpora produce better grounded answers than generic text.",
    "Product documentation and FAQs are common sources for RAG knowledge bases.",
    "Chunk overlap can improve recall but may increase redundancy.",
    "Embedding model choice affects semantic recall and precision.",
    "Vector search can power semantic search, Q&A, and recommendations.",
    "Oracle Vector Search supports cosine and L2 distance metrics for similarity.",
    "To build a RAG system with Oracle and Claude, first ingest documents into Oracle AI Database as vectors.",
    "Each document is embedded using an embedding model and stored in a VECTOR column.",
    "At query time, the user question is embedded and used for similarity search in Oracle Vector Search.",
    "The top-k retrieved documents are passed as context to Claude for grounded generation.",
    "Claude generates the final answer based only on the retrieved context.",
    "RAG systems combine retrieval from a vector database with LLM generation to reduce hallucinations.",
    "Oracle AI Database provides native VECTOR types and similarity search for RAG pipelines.",
    "Python applications can orchestrate ingestion, retrieval, and LLM calls in a RAG workflow."
]
print("Documents loaded:", len(docs))


Documents loaded: 28


### Step 6: Generate embeddings

We convert each document into a numeric vector using a local embedding model 
from SentenceTransformers.

This avoids relying on closed embedding APIs and makes the cookbook easier 
to reproduce for everyone.

Claude is used only for generation (LLM), not for embeddings.


In [7]:
from sentence_transformers import SentenceTransformer
import logging
import os
import warnings

logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

embedder = SentenceTransformer("intfloat/e5-large-v2")

def embed(text: str):
    return embedder.encode(text, normalize_embeddings=True, show_progress_bar=False).tolist()

v = embed("Oracle Vector Search with Claude")
print("Embedding model loaded. Vector dimension =", len(v))


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 582.99it/s, Materializing param=pooler.dense.weight]                               


Embedding model loaded. Vector dimension = 1024


### Step 7: Ingest documents into Oracle AI Database

We generate embeddings for each document and insert them into Oracle AI Database.

Oracle expects vectors in binary format, so we convert Python lists into `array("f")` before inserting.


In [8]:
from array import array

for d in docs:
    emb = embed(d)

    assert len(emb) == 1024, f"Vector dim mismatch: {len(emb)}"

    vec = array("f", emb)

    cursor.execute(
        "INSERT INTO documents (content, embedding) VALUES (:1, :2)",
        [d, vec]
    )

conn.commit()
print("Documents ingested OK")


Documents ingested OK


### Step 8: Perform similarity search

We embed the user query and retrieve the top-K most similar documents using Oracle AI Database vector similarity search.

This step represents the **retrieval** part of RAG.


In [9]:
query = "How do I build a RAG system with Oracle and Claude?"
q_emb = embed(query)
q_vec = array("f", q_emb)

cursor.execute("""
    SELECT content
    FROM documents
    ORDER BY embedding <-> :1
    FETCH FIRST 2 ROWS ONLY
""", [q_vec])

rows = cursor.fetchall()

unique_chunks = []
for r in rows:
    text = r[0].read() if hasattr(r[0], "read") else r[0]
    if text not in unique_chunks:
        unique_chunks.append(text)

context = "\n---\n".join(unique_chunks)

print("Retrieved context:\n", context)



Retrieved context:
 To build a RAG system with Oracle and Claude, first ingest documents into Oracle AI Database as vectors.
---
Claude is used for generation and reasoning, while embeddings can be produced by a separate model.


### Step 9: Generate a grounded answer with Claude

We pass the retrieved context to Claude and instruct it to answer **only using the provided documents**.

This reduces hallucinations and ensures answers are grounded in your data.

In [10]:
from anthropic import Anthropic
import os

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))


### Model selection

Recommended (requires access on Anthropic account):
- Claude Sonnet 4.5 (best quality)
- MODEL_NAME = "claude-4.5-sonnet"

Fallback (publicly available model)
- Uncomment this if you don't have access to 4.5 yet
- MODEL_NAME = "claude-3-5-sonnet-20240620"

In [11]:
resp = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": f"""
You are a helpful assistant.

Context:
{context}

Question:
{query}

Answer clearly and concisely based only on the context.
"""
        }
    ]
)

print("Final answer:\n", resp.content[0].text)


Final answer:
 Based on the provided context, to build a RAG (Retrieval-Augmented Generation) system with Oracle and Claude, the following steps can be followed:

1. Ingest documents into Oracle AI Database as vectors:
   - Use the Oracle AI Database to store the documents that will be used for the RAG system.
   - Convert the documents into vector representations and store them in the database.

2. Use Claude for generation and reasoning:
   - Utilize the Claude model for the generation and reasoning components of the RAG system.
   - Claude can be used to generate responses or perform reasoning tasks based on the retrieved information from the Oracle AI Database.

3. Use a separate model for embeddings:
   - A separate model, different from Claude, can be used to produce the embeddings for the documents stored in the Oracle AI Database.
   - This separate model will be responsible for generating the vector representations of the documents.

The key aspects of building the RAG system 

### Inspect retrieval quality

We inspect the retrieved results and similarity scores.

This helps you:

- debug retrieval quality  
- tune chunking strategies  
- evaluate embedding performance  

In real systems, this step is useful for observability and evaluation.


⚠️ Note: The similarity scores shown below are mocked values for visualization purposes.
In a real application, you would retrieve similarity scores directly from Oracle Vector Search queries.


In [12]:
import pandas as pd

results = [
    {"content": d, "score": s}
    for d, s in zip(docs, [0.89, 0.82, 0.79])
]

df = pd.DataFrame(results)
df


,content,score
0,Oracle AI Database provides native vector sear...,0.89
1,Vector Search enables similarity queries over ...,0.82
2,RAG systems combine retrieval with LLMs to red...,0.79


### Step 10: Cleanup resources

In [13]:
try:
    cursor.close()
    conn.close()
    print("Oracle DB connection closed.")
except Exception as e:
    print("Cleanup error:", e)

Oracle DB connection closed.
